In [1]:
# Phase I-V Pipeline for Narrative Page Reconstruction (train3.ipynb)
# - Phase I: normalization + NER + stylometric clustering
# - Phase II: cross-encoder pairwise continuation scoring
# - Phase III: global sequence with ATSP
# - Phase IV: optional open-source LLM local reranking
# - Phase V: output validation + CSV submission

from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Iterable
import itertools
import math
import random
import re

import numpy as np
import pandas as pd
from scipy.stats import kendalltau
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sentence_transformers import SentenceTransformer, CrossEncoder, InputExample
from torch.utils.data import DataLoader
from ortools.constraint_solver import routing_enums_pb2, pywrapcp

# ------------------------------
# Config
# ------------------------------

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

BASE = Path(r"/teamspace/studios/this_studio/QA")
DATA_DIR = BASE / "data"
OUT_DIR = BASE / "submissions"
OUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = DATA_DIR / "Mysterious_Affair_at_Styles_Train_Data.csv"
BOOK_A_PATH = DATA_DIR / "BookA_test.csv"
BOOK_B_PATH = DATA_DIR / "BookB_test.csv"

BOOK_A_OUT = OUT_DIR / "BookA_train3.csv"
BOOK_B_OUT = OUT_DIR / "BookB_train3.csv"

# RoBERTa-based encoders
BI_ENCODER_NAME = "sentence-transformers/all-roberta-large-v1"
CROSS_ENCODER_NAME = "cross-encoder/stsb-roberta-base"

# Keep training light by default; increase for stronger results.
DO_FINE_TUNE = True
EPOCHS = 1
BATCH_SIZE = 8
HARD_NEG_PER_POS = 2

# Optional LLM coherence refinement (off by default, can be expensive).
ENABLE_LLM_REFINEMENT = False
LLM_MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2"
LLM_WINDOW = 8
MAX_SWAP_RADIUS = 2
MAX_REFINEMENT_PASSES = 2

# ------------------------------
# Utilities
# ------------------------------

def clean_text(text: str) -> str:
    txt = "" if pd.isna(text) else str(text)
    txt = txt.replace("\u2019", "'").replace("\u201c", '"').replace("\u201d", '"')
    txt = txt.replace("\u2014", "-")
    txt = re.sub(r"\s+", " ", txt).strip()
    return txt

def split_sentences(text: str) -> list[str]:
    t = clean_text(text)
    return [s.strip() for s in re.split(r"(?<=[.!?])\s+", t) if s.strip()]

def head_snippet(text: str, n_sentences: int = 3, max_chars: int = 900) -> str:
    sents = split_sentences(text)
    snippet = " ".join(sents[:n_sentences]) if sents else clean_text(text)
    return snippet[:max_chars]

def tail_snippet(text: str, n_sentences: int = 3, max_chars: int = 900) -> str:
    sents = split_sentences(text)
    snippet = " ".join(sents[-n_sentences:]) if sents else clean_text(text)
    return snippet[-max_chars:]

def roman_to_int(s: str) -> int:
    vals = {"I": 1, "V": 5, "X": 10, "L": 50, "C": 100, "D": 500, "M": 1000}
    s = s.upper()
    total = 0
    prev = 0
    for ch in reversed(s):
        v = vals.get(ch, 0)
        if v == 0:
            return 0
        if v < prev:
            total -= v
        else:
            total += v
            prev = v
    return total

def extract_chapter_number(text: str) -> int:
    t = clean_text(text)
    m = re.match(r"(?i)^\s*(?:chapter|part|book)?\s*[:.-]?\s*([ivxlcdm]+|\d+)\b", t)
    if not m:
        return 0
    tok = m.group(1)
    return int(tok) if tok.isdigit() else roman_to_int(tok)

def lexical_richness(text: str) -> float:
    words = re.findall(r"[A-Za-z]+", clean_text(text).lower())
    if not words:
        return 0.0
    return len(set(words)) / len(words)

def avg_sentence_length(text: str) -> float:
    sents = split_sentences(text)
    if not sents:
        return 0.0
    return float(np.mean([len(re.findall(r"[A-Za-z]+", s)) for s in sents]))

def punctuation_density(text: str) -> float:
    t = clean_text(text)
    if not t:
        return 0.0
    punct = len(re.findall(r"[,:;.!?\-\"'()]", t))
    return punct / max(len(t), 1)

def style_feature_vector(text: str) -> list[float]:
    t = clean_text(text)
    return [
        len(t),
        avg_sentence_length(t),
        lexical_richness(t),
        punctuation_density(t),
        float(bool(re.search(r"\b(?:Mr|Mrs|Miss|Dr|Sir)\.?\b", t))),
    ]

# ------------------------------
# Phase I: NER + Stylometric Clustering
# ------------------------------

def _load_spacy_ner():
    try:
        import spacy
        try:
            return spacy.load("en_core_web_trf")
        except Exception:
            return spacy.load("en_core_web_sm")
    except Exception:
        return None

NER_NLP = _load_spacy_ner()

def extract_entities(text: str) -> dict[str, list[str]]:
    t = clean_text(text)
    out = {"PERSON": [], "GPE": [], "DATE": [], "TIME": [], "OBJECT": []}
    if not t:
        return out

    if NER_NLP is not None:
        doc = NER_NLP(t)
        for ent in doc.ents:
            label = ent.label_
            if label == "PERSON":
                out["PERSON"].append(ent.text)
            elif label in {"GPE", "LOC", "FAC"}:
                out["GPE"].append(ent.text)
            elif label == "DATE":
                out["DATE"].append(ent.text)
            elif label == "TIME":
                out["TIME"].append(ent.text)
            elif label in {"PRODUCT", "WORK_OF_ART", "ORG"}:
                out["OBJECT"].append(ent.text)
    else:
        # Regex fallback if spaCy model isn't available.
        out["PERSON"] = re.findall(r"\b(?:Mr|Mrs|Miss|Dr|Sir)\.?\s+[A-Z][a-z]+\b", t)
        out["DATE"] = re.findall(r"\b(?:Monday|Tuesday|Wednesday|Thursday|Friday|Saturday|Sunday|\d{1,2}(?:st|nd|rd|th)?\s+[A-Z][a-z]+)\b", t)
        out["TIME"] = re.findall(r"\b\d{1,2}:\d{2}\s*(?:a\.m\.|p\.m\.|AM|PM)?\b", t)
        out["GPE"] = re.findall(r"\b(?:London|England|Styles|Essex)\b", t)

    for k in out:
        out[k] = sorted(set(out[k]))
    return out

def build_entity_matrix(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for _, row in df.iterrows():
        page_id = int(row["page"] if "page" in df.columns else row.name + 1)
        ents = extract_entities(str(row["text"]))
        rows.append(
            {
                "page": page_id,
                "person_n": len(ents["PERSON"]),
                "gpe_n": len(ents["GPE"]),
                "date_n": len(ents["DATE"]),
                "time_n": len(ents["TIME"]),
                "object_n": len(ents["OBJECT"]),
            }
        )
    return pd.DataFrame(rows)

print("Loading datasets...")
train_df = pd.read_csv(TRAIN_PATH)
book_a_df = pd.read_csv(BOOK_A_PATH)
book_b_df = pd.read_csv(BOOK_B_PATH)

for d in (train_df, book_a_df, book_b_df):
    d["text"] = d["text"].map(clean_text)

print("Encoding pages with RoBERTa bi-encoder...")
bi_encoder = SentenceTransformer(BI_ENCODER_NAME)

def build_representation(df: pd.DataFrame) -> dict[str, np.ndarray | list[str] | pd.DataFrame]:
    texts = df["text"].tolist()
    heads = [head_snippet(t) for t in texts]
    tails = [tail_snippet(t) for t in texts]

    full_emb = bi_encoder.encode(texts, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=False)
    head_emb = bi_encoder.encode(heads, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=False)
    tail_emb = bi_encoder.encode(tails, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=False)

    style_feats = np.array([style_feature_vector(t) for t in texts], dtype=np.float32)
    entity_df = build_entity_matrix(df)
    entity_feats = entity_df[["person_n", "gpe_n", "date_n", "time_n", "object_n"]].to_numpy(dtype=np.float32)
    chapter_num = np.array([extract_chapter_number(t) for t in texts], dtype=np.float32).reshape(-1, 1)

    page_feats = np.hstack([full_emb, head_emb, tail_emb, style_feats, entity_feats, chapter_num])

    return {
        "texts": texts,
        "heads": heads,
        "tails": tails,
        "full_emb": full_emb,
        "head_emb": head_emb,
        "tail_emb": tail_emb,
        "style_feats": style_feats,
        "entity_df": entity_df,
        "page_feats": page_feats,
    }

train_rep = build_representation(train_df)
book_a_rep = build_representation(book_a_df)
book_b_rep = build_representation(book_b_df)

print("Running unsupervised clustering (KMeans + DBSCAN)...")
all_feats = np.vstack([train_rep["page_feats"], book_a_rep["page_feats"], book_b_rep["page_feats"]])
scaler = StandardScaler()
scaled_all = scaler.fit_transform(all_feats)

pca_dim = min(64, scaled_all.shape[1], scaled_all.shape[0] - 1)
pca = PCA(n_components=pca_dim, random_state=SEED)
pca_all = pca.fit_transform(scaled_all)

kmeans = KMeans(n_clusters=6, random_state=SEED, n_init="auto")
kmeans_labels = kmeans.fit_predict(pca_all)
dbscan = DBSCAN(eps=2.0, min_samples=3)
dbscan_labels = dbscan.fit_predict(pca_all)

print("KMeans clusters:", np.unique(kmeans_labels, return_counts=True))
print("DBSCAN clusters:", np.unique(dbscan_labels, return_counts=True))

# ------------------------------
# Phase II: Cross-Encoder Pairwise Scoring
# ------------------------------

def build_train_pairs(df: pd.DataFrame, hard_neg_per_pos: int = 2) -> list[InputExample]:
    texts = df["text"].tolist()
    n = len(texts)
    examples: list[InputExample] = []

    # Positive adjacent pairs
    for i in range(n - 1):
        examples.append(InputExample(texts=[tail_snippet(texts[i]), head_snippet(texts[i + 1])], label=1.0))

    # Hard negatives: near-context but non-adjacent
    for i in range(n - 1):
        candidates = []
        for j in range(n):
            if j in {i, i + 1}:
                continue
            if abs(j - (i + 1)) <= 6:
                candidates.append(j)
        if not candidates:
            continue
        picked = random.sample(candidates, k=min(hard_neg_per_pos, len(candidates)))
        for j in picked:
            examples.append(InputExample(texts=[tail_snippet(texts[i]), head_snippet(texts[j])], label=0.0))

    random.shuffle(examples)
    return examples

print("Preparing cross-encoder...")
cross_encoder = CrossEncoder(CROSS_ENCODER_NAME, num_labels=1)

if DO_FINE_TUNE:
    print("Fine-tuning cross-encoder...")
    train_examples = build_train_pairs(train_df, hard_neg_per_pos=HARD_NEG_PER_POS)
    train_loader = DataLoader(train_examples, shuffle=True, batch_size=BATCH_SIZE)
    cross_encoder.fit(
        train_dataloader=train_loader,
        epochs=EPOCHS,
        warmup_steps=max(1, len(train_loader) // 10),
        show_progress_bar=True,
    )

def build_transition_matrix(rep: dict[str, np.ndarray | list[str]], batch_size: int = 32) -> np.ndarray:
    heads = rep["heads"]
    tails = rep["tails"]
    n = len(heads)
    mat = np.zeros((n, n), dtype=np.float32)

    pairs = []
    index_pairs = []
    for i in range(n):
        for j in range(n):
            if i == j:
                continue
            pairs.append((tails[i], heads[j]))
            index_pairs.append((i, j))

    scores = cross_encoder.predict(pairs, batch_size=batch_size, show_progress_bar=True)
    probs = 1.0 / (1.0 + np.exp(-np.asarray(scores, dtype=np.float32)))

    for (i, j), p in zip(index_pairs, probs):
        mat[i, j] = float(np.clip(p, 1e-6, 1.0 - 1e-6))

    np.fill_diagonal(mat, 1e-6)
    return mat

# ------------------------------
# Phase III: ATSP Global Solve
# ------------------------------

def probability_to_cost(prob_matrix: np.ndarray) -> np.ndarray:
    p = np.clip(prob_matrix, 1e-6, 1.0)
    return -np.log(p)

def solve_atsp_open_path(cost_matrix: np.ndarray, time_limit_seconds: int = 30) -> list[int]:
    n = cost_matrix.shape[0]
    aug = np.pad(cost_matrix, ((0, 1), (0, 1)), mode="constant", constant_values=0.0)

    scale = 100000
    int_cost = np.rint(aug * scale).astype(np.int64)

    manager = pywrapcp.RoutingIndexManager(n + 1, 1, n)
    routing = pywrapcp.RoutingModel(manager)

    def distance_callback(from_idx: int, to_idx: int) -> int:
        i = manager.IndexToNode(from_idx)
        j = manager.IndexToNode(to_idx)
        return int(int_cost[i, j])

    transit_idx = routing.RegisterTransitCallback(distance_callback)
    routing.SetArcCostEvaluatorOfAllVehicles(transit_idx)

    search = pywrapcp.DefaultRoutingSearchParameters()
    search.first_solution_strategy = routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
    search.local_search_metaheuristic = routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
    search.time_limit.seconds = time_limit_seconds

    sol = routing.SolveWithParameters(search)
    if sol is None:
        return []

    order = []
    idx = routing.Start(0)
    while not routing.IsEnd(idx):
        node = manager.IndexToNode(idx)
        if node != n:
            order.append(node)
        idx = sol.Value(routing.NextVar(idx))
    return order

# ------------------------------
# Phase IV: Optional LLM Refinement
# ------------------------------

def _load_llm_pipeline():
    if not ENABLE_LLM_REFINEMENT:
        return None
    try:
        from transformers import pipeline
        return pipeline("text-generation", model=LLM_MODEL_NAME)
    except Exception as ex:
        print("LLM pipeline not available, skipping refinement:", ex)
        return None

llm_pipe = _load_llm_pipeline()

def local_coherence_penalty(seq_texts: list[str], i: int, j: int) -> float:
    if llm_pipe is None:
        return 0.0
    block = "\n\n".join(seq_texts[i:j])
    prompt = (
        "Rate chronological coherence from 0 (perfect) to 10 (broken).\n"
        "Return only a number.\n\n"
        f"{block}"
    )
    out = llm_pipe(prompt, max_new_tokens=4, do_sample=False)[0]["generated_text"]
    m = re.search(r"(\d+(?:\.\d+)?)", out)
    return float(m.group(1)) if m else 5.0

def refine_order_with_local_swaps(order: list[int], texts: list[str]) -> list[int]:
    if llm_pipe is None or len(order) < 4:
        return order

    best = order[:]
    best_texts = [texts[i] for i in best]

    for _ in range(MAX_REFINEMENT_PASSES):
        improved = False
        for center in range(1, len(best) - 1):
            l = max(0, center - LLM_WINDOW // 2)
            r = min(len(best), center + LLM_WINDOW // 2)
            base_pen = local_coherence_penalty(best_texts, l, r)

            for d in range(1, MAX_SWAP_RADIUS + 1):
                i, j = center, min(center + d, len(best) - 1)
                cand = best[:]
                cand[i], cand[j] = cand[j], cand[i]
                cand_texts = [texts[k] for k in cand]
                cand_pen = local_coherence_penalty(cand_texts, l, r)
                if cand_pen + 1e-6 < base_pen:
                    best = cand
                    best_texts = cand_texts
                    improved = True
                    break
            if improved:
                break
        if not improved:
            break

    return best

# ------------------------------
# Phase V: Validation + Submission
# ------------------------------

def validate_submission(df_in: pd.DataFrame, order: list[int]) -> tuple[bool, str]:
    n = len(df_in)
    if len(order) != n or len(set(order)) != n:
        return False, "Order is not a full permutation."

    original_page = np.arange(1, n + 1, dtype=int)
    shuffled_page = df_in.iloc[order]["page"].to_numpy() if "page" in df_in.columns else np.array(order) + 1

    if not np.array_equal(original_page, np.arange(1, n + 1, dtype=int)):
        return False, "original_page is not continuous from 1..N."

    in_set = set(df_in["page"].tolist()) if "page" in df_in.columns else set(range(1, n + 1))
    out_set = set(map(int, shuffled_page))
    if in_set != out_set:
        return False, "shuffled_page is not an exact permutation of source pages."

    return True, "OK"

def save_submission(df_in: pd.DataFrame, order: list[int], out_path: Path) -> None:
    sub = pd.DataFrame(
        {
            "original_page": np.arange(1, len(order) + 1, dtype=int),
            "shuffled_page": df_in.iloc[order]["page"].to_numpy(dtype=int),
        }
    )
    sub.to_csv(out_path, index=False)

def reconstruct_book(df: pd.DataFrame, rep: dict[str, np.ndarray | list[str]], name: str) -> list[int]:
    print(f"Building transition matrix for {name}...")
    prob = build_transition_matrix(rep)
    cost = probability_to_cost(prob)
    order = solve_atsp_open_path(cost, time_limit_seconds=45)

    if not order:
        print(f"ATSP fallback for {name}: using argmax chain heuristic")
        n = prob.shape[0]
        used = set()
        cur = int(np.argmax(prob.sum(axis=1)))
        order = [cur]
        used.add(cur)
        while len(order) < n:
            nxt = int(np.argmax(np.where(np.isin(np.arange(n), list(used)), -1.0, prob[cur])))
            if nxt in used:
                nxt = next(i for i in range(n) if i not in used)
            order.append(nxt)
            used.add(nxt)
            cur = nxt

    order = refine_order_with_local_swaps(order, rep["texts"])

    ok, msg = validate_submission(df, order)
    print(f"Validation for {name}: {msg}")
    if not ok:
        raise ValueError(f"Submission validation failed for {name}: {msg}")
    return order

print("Reconstructing Book A...")
book_a_order = reconstruct_book(book_a_df, book_a_rep, "BookA")
save_submission(book_a_df, book_a_order, BOOK_A_OUT)

print("Reconstructing Book B...")
book_b_order = reconstruct_book(book_b_df, book_b_rep, "BookB")
save_submission(book_b_df, book_b_order, BOOK_B_OUT)

# Local self-shuffle validation on training corpus
def kendall_on_self_shuffle(df: pd.DataFrame) -> float:
    shuffled = df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
    rep = build_representation(shuffled)
    order = reconstruct_book(shuffled, rep, "TrainSelfShuffle")

    predicted_pos = np.empty(len(shuffled), dtype=int)
    for rank, idx in enumerate(order, start=1):
        predicted_pos[idx] = rank
    true_pos = shuffled["page"].to_numpy(dtype=int)
    tau = kendalltau(true_pos, predicted_pos).correlation
    return float(tau)

print("Running local validation...")
train_tau = kendall_on_self_shuffle(train_df)
print(f"Train self-shuffle Kendall tau: {train_tau:.4f}")
print(f"Saved: {BOOK_A_OUT}")
print(f"Saved: {BOOK_B_OUT}")

Loading datasets...
Encoding pages with RoBERTa bi-encoder...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: sentence-transformers/all-roberta-large-v1
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Running unsupervised clustering (KMeans + DBSCAN)...
KMeans clusters: (array([0, 1, 2, 3, 4, 5], dtype=int32), array([ 73,  66,  16,  53,  76, 127]))
DBSCAN clusters: (array([-1]), array([411]))
Preparing cross-encoder...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

...

Validation for BookA: OK
Reconstructing Book B...
Building transition matrix for BookB...


Batches:   0%|          | 0/100 [00:00<?, ?it/s]

Validation for BookB: OK
Running local validation...
Building transition matrix for TrainSelfShuffle...


Batches:   0%|          | 0/1333 [00:00<?, ?it/s]

Validation for TrainSelfShuffle: OK
Train self-shuffle Kendall tau: -0.0389
Saved: /teamspace/studios/this_studio/QA/submissions/BookA_train3.csv
Saved: /teamspace/studios/this_studio/QA/submissions/BookB_train3.csv
